# E1.4 · Control mapping for agents

**Function E — AI Governance for Agentic Systems → Building the Governance Framework — Risk and Control**  ·  *Security of AI*

Builds on **[E1.3 · Risk tiering agentic use cases](https://spbreed.github.io/cyber-commons/lessons/E1.3.html)**.

| | |
|---|---|
| Tools used | OSCAL |

## What this lesson is

**What it covers.** Map the A2/A3 controls onto your control library.

**Why a security engineer needs it.** Inventing new controls where an existing one applied to a new principal type. The control it builds is: map identity, secrets, sandbox, eval and telemetry onto the existing library.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

Most agentic risks map onto controls you already have. Building a second, parallel control estate for AI is the most common and most expensive mistake in this function — the work is finding the genuine gaps, not restating the overlap.

> **At CyberTravels.** Most of CyberTravels' risks map onto controls it already has. Building a parallel AI control estate is the expensive mistake; finding the genuine gaps is the work.

## 2 · The framework

```
   agentic risk               existing control          gap?
   +-------------------+      +------------------+      +-----+
   | prompt injection  |      | input validation |      | yes |
   | over-privilege    |      | least privilege  |      | no  |
   | no attribution    |      | audit logging    |      | yes |
   +-------------------+      +------------------+      +-----+

   the work is finding the "yes" rows, not restating the "no" ones
```

Control mapping runs one way: **control → framework.**

Starting from the framework produces a checklist that is complete, satisfies an
assessor, and defends nothing — because it enumerates clauses rather than
capabilities, and a clause with no operating control behind it evidences nothing.

Starting from controls produces the opposite: a smaller list of things you
actually do, each of which happens to satisfy several framework clauses. The
framework coverage is an **output**, and that is the only mapping that survives a
supervisor asking "show me".

## 3 · The procedure, as a skill

The skill maps eight operating controls outward to clauses across five frameworks, attaches the evidence artefact each control would be shown by, and derives what a critical tier requires — so coverage comes out as an output rather than a claim.

In [ ]:
# skills/grc/control-to-framework-mapping/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: control-to-framework-mapping
description: >-
  Map a control catalogue outward to framework clauses — never the reverse — and
  assemble the evidence pack a given risk tier requires. Use when a framework
  checklist is being turned into a programme, or when coverage is claimed
  without artefacts.
allowed-tools: Read, Grep, Glob
---

# Control to framework, never framework to control

Starting from the framework produces a checklist that is complete, satisfies an
assessor, and defends nothing: it enumerates clauses rather than capabilities,
and a clause with no operating control behind it evidences nothing. Starting
from controls produces a smaller list of things you actually do, each of which
happens to satisfy several clauses.

## When to use this

Building a control catalogue, responding to a framework mapping request, or
auditing a coverage claim.

## Procedure

**1 — Write the catalogue first, as operating controls.** Each one a mechanism
somebody runs, with an owner. If a row cannot be described as something that
runs, it is a policy statement and belongs elsewhere.

**2 — Map each control outward.** One control to many clauses across every
framework you report against. The one-to-many direction is what makes this
cheaper than it looks.

**3 — Attach the evidence artefact per control.** A log, an export, a test
result, a signed attestation — the thing an assessor would be handed. A control
with no artefact is unevidenced whatever its status says.

**4 — Derive requirements per tier.** Which controls a critical-tier system must
have, which a medium one. Then count the clauses that follow, rather than
promising clause coverage directly.

**5 — Report coverage as an output.** "These 8 controls satisfy 12 clauses" is
defensible; "we cover 12 clauses" invites the follow-up you cannot answer.

## Output contract

```json
{
  "catalogue": [{"id": "str", "control": "str", "owner": "str", "artefact": "str"}],
  "mapping": [{"control": "str", "framework": "str", "clauses": ["str"]}],
  "tiers": [{"tier": "str", "requires": ["str"], "clauses_satisfied": 0}],
  "coverage": {"controls": 0, "clauses": 0, "unevidenced": ["str"]}
}
```

## Failure modes

- **Starting from the framework.** You get a checklist, not a programme.
- **A control with no artefact.** It cannot be shown.
- **Claiming clause coverage directly.** The assessor asks which control, and
  there is no answer.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/grc/control-to-framework-mapping/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/grc/control-to-framework-mapping/scripts/control_to_framework_mapping.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Map a control catalogue outward to framework clauses, and assemble the evidence pack a tier requires.

This is the executable half of the `control-to-framework-mapping` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

from dataclasses import dataclass

@dataclass(frozen=True)
class Control:
    cid: str; text: str; kind: str; frameworks: tuple

CATALOGUE = [
 Control("AC-1", "agent identities are distinct from human and separately revocable",
         "preventive", ("NIST AI RMF: GOVERN-1.2", "ISO 42001: 6.1", "EU AI Act: Art.14")),
 Control("AC-2", "delegated authority narrows at every hop and is recorded in an act chain",
         "preventive", ("NIST AI RMF: MANAGE-2.2", "ISO 42001: 8.1")),
 Control("SB-1", "agent egress is deny-by-default with an allowlist",
         "preventive", ("NIST AI RMF: MANAGE-2.1", "ISO 27001: A.8.20")),
 Control("SB-2", "privileged tools require approval below autonomy L3",
         "preventive", ("EU AI Act: Art.14 human oversight",)),
 Control("EV-1", "every agent action is logged with the acting identity",
         "detective", ("ISO 42001: 9.1", "EU AI Act: Art.12 record-keeping")),
 Control("EV-2", "harness accuracy evaluated against a held-out key each release",
         "detective", ("NIST AI RMF: MEASURE-2.3",)),
 Control("DR-1", "behavioural drift from the signed-off baseline raises an alert",
         "detective", ("NIST AI RMF: MEASURE-2.4", "ISO 42001: 9.1")),
 Control("ST-1", "a tested stop mechanism halts an agent fleet without vendor help",
         "corrective", ("EU AI Act: Art.14", "DORA: Art.11")),
]
print(f"{'control':8s}{'kind':12s}satisfies")
print("-" * 84)
for c in CATALOGUE:
    print(f"{c.cid:8s}{c.kind:12s}{len(c.frameworks)} clause(s): {c.frameworks[0]}")
    for f in c.frameworks[1:]:
        print(f"{'':20s}{f}")

def map_controls(tier, catalogue=CATALOGUE):
    if tier in ("critical", "high"):
        required = list(catalogue)
    else:
        required = [c for c in catalogue if c.kind == "preventive" or c.cid == "EV-1"]
    frameworks = sorted({f for c in required for f in c.frameworks})
    return {"tier": tier, "controls": [c.cid for c in required],
            "frameworks_satisfied": frameworks}

for tier in ("critical", "medium"):
    m = map_controls(tier)
    print(f"\ntier {tier}: {len(m['controls'])} controls → "
          f"{len(m['frameworks_satisfied'])} framework clauses")
    print(f"   controls   {m['controls']}")
    for f in m["frameworks_satisfied"]:
        print(f"   satisfies  {f}")

FRAMEWORK_CLAUSES = [
 "NIST AI RMF: GOVERN-1.1 policies are documented",
 "NIST AI RMF: GOVERN-1.2 roles and responsibilities are defined",
 "NIST AI RMF: MAP-1.1 context is established",
 "NIST AI RMF: MEASURE-2.3 performance is evaluated",
 "ISO 42001: 6.1 actions to address risks",
 "ISO 42001: 7.2 competence",
 "ISO 42001: 9.1 monitoring and measurement",
]
have = {f for c in CATALOGUE for f in c.frameworks}
print(f"{'clause':52s}{'operating control?':>20}")
print("-" * 74)
orphans = []
for clause in FRAMEWORK_CLAUSES:
    covered = clause in have
    if not covered: orphans.append(clause)
    print(f"{clause:52s}{('yes' if covered else 'NO — checklist only'):>20}")
print(f"\n{len(orphans)}/{len(FRAMEWORK_CLAUSES)} clauses have no operating control behind them.")
print("Working framework-first, those get a policy document and a tick. Working")
print("control-first, they are visibly uncovered — which is the useful state.")
assert orphans

EVIDENCE = {
 "AC-1": "gateway logs containing an act chain for every action; monthly sample",
 "AC-2": "regression suite cases IDN-01/IDN-04, run on every release",
 "SB-1": "90-day egress denial log",
 "SB-2": "tool policy in git + denial log",
 "EV-1": "audit sample of 50 actions with acting identity present",
 "EV-2": "expert accuracy against a held-out key, per release",
 "DR-1": "drift alerts and their dispositions",
 "ST-1": "game-day record with measured time-to-stop",
}
def evidence_pack(tier):
    m = map_controls(tier)
    return [{"control": cid, "evidence": EVIDENCE[cid],
             "satisfies": [f for c in CATALOGUE if c.cid == cid for f in c.frameworks]}
            for cid in m["controls"]]

pack = evidence_pack("critical")
for row in pack[:4]:
    print(f"{row['control']}  {row['evidence']}")
    for f in row["satisfies"]:
        print(f"      → {f}")
print(f"\n{len(pack)} controls produce evidence for "
      f"{len({f for r in pack for f in r['satisfies']})} framework clauses.")
print("One artefact, many clauses. That ratio is why control-first is cheaper.")

## What you just proved

The catalogue's 8 controls map to framework clauses across NIST AI RMF, ISO 42001, ISO 27001, the EU AI Act and DORA. Critical tier requires all 8 and satisfies 12 clauses; medium requires 5. Working framework-first leaves 3 of 7 clauses with no operating control. The evidence pack shows one artefact satisfying several clauses.

## Your turn

Take one framework clause your programme claims to satisfy and ask which operating control produces its evidence. If the answer is a policy document, the clause is ticked and undefended.

---

**Next → [E1.5 · Evaluation output as audit evidence](https://spbreed.github.io/cyber-commons/lessons/E1.5.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/E1.4.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/E1.4.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*